# Week 2 — GROUP BY, Aggregates, HAVING: Summarising Data by Group
## Phase 2b SQL | PORA Academy Cohort 7 — **Exercises**

Last week you selected and filtered individual rows. This week you collapse many rows into **one row per group** with `GROUP BY`, summarise each group with `COUNT`, `SUM`, `AVG`, `MIN` and `MAX`, and then filter the *groups themselves* with `HAVING`.

Each question below comes as **three cells**:

1. A **question** with the task and an **Expected** result.
2. A blank `%%sql` answer cell — write your query where it says `-- Your query here`, capturing the result into a variable (e.g. `q1`).
3. A **check cell** (plain Python) — run it after your query. A ✅ means you got it right, and the cell then displays the table your query returned.

**Do not edit the check cells.** Run the setup cell first, then work top to bottom.

Reminders for this week:
- Every non-aggregated column in the `SELECT` list must also appear in `GROUP BY`.
- `WHERE` filters **rows before** grouping; `HAVING` filters **groups after** grouping.
- Alias your aggregates exactly as each question asks (e.g. `AS n`) — the check cells look up those column names.
- Wrap averages and sums in `ROUND(..., 2)` when a question asks for 2 decimal places.


In [ ]:
# =====================================================================
# Olist SQL Setup — runs on BOTH Google Colab and a local machine.
# Run this cell FIRST. It loads the 8 Olist tables into a SQLite
# database and connects the %%sql magic to it. You should not need to
# edit anything unless auto-detection fails (see the two knobs below).
#
# Design notes:
# - We teach SQL with the %%sql cell magic (jupysql), not pd.read_sql().
# - jupysql opens its OWN connection, so the DB must be a real FILE
#   (a :memory: DB would be invisible to it).
# - We use jupysql (the maintained SQL magic). On Colab we install it,
#   because Colab ships the legacy ipython-sql, which (a) can't take a
#   connection by engine variable and (b) renders every result through
#   prettytable.__dict__[style], crashing on modern prettytable with
#   KeyError 'DEFAULT'/'SINGLE_BORDER'. jupysql fixes both.
# - autopandas=True makes every %%sql result a pandas DataFrame, which
#   lets the self-check cells assert on .iloc/.shape directly.
# =====================================================================
import os, glob, sqlite3, tempfile, zipfile
import pandas as pd

# --- Optional knobs (leave blank; only set if auto-detect fails) ------
LOCAL_DATA_DIR = ""   # local run: folder that holds olist_orders_dataset.csv
DRIVE_ZIP_PATH = ""   # Colab: full path to phase-2-python-sql.zip in your Drive
# ---------------------------------------------------------------------

# Detect Colab (google.colab only imports there). Outside Colab — including
# the content-pipeline validator — this falls through to the local branch.
try:
    from google.colab import drive
    drive.mount("/content/drive")
    ON_COLAB = True
except ModuleNotFoundError:
    ON_COLAB = False


def _colab_find_zip():
    """Locate phase-2-python-sql.zip in Drive WITHOUT a full recursive scan
    (globbing '/content/drive/MyDrive/**' walks the entire Drive over the
    network and can hang for many minutes). Try explicit paths first, then a
    depth- and count-bounded breadth-first search that prints progress."""
    if DRIVE_ZIP_PATH:
        if os.path.exists(DRIVE_ZIP_PATH):
            return DRIVE_ZIP_PATH
        raise FileNotFoundError(f"DRIVE_ZIP_PATH is set but not found: {DRIVE_ZIP_PATH}")

    target = "phase-2-python-sql.zip"
    # Fast, instant checks of the most likely spots (top of Drive + course folder).
    for cand in (
        f"/content/drive/MyDrive/{target}",
        f"/content/drive/MyDrive/Data Analysis and AI Automation Course Cohort 7/Dataset/{target}",
        f"/content/{target}",
    ):
        if os.path.exists(cand):
            return cand

    # Bounded BFS: depth <= 4, at most ~600 folders, skipping hidden dirs.
    print("Searching your Google Drive for phase-2-python-sql.zip ...")
    root, queue, scanned = "/content/drive/MyDrive", [("/content/drive/MyDrive", 0)], 0
    while queue:
        d, depth = queue.pop(0)
        hit = os.path.join(d, target)
        if os.path.exists(hit):
            return hit
        if depth >= 4:
            continue
        try:
            for e in os.scandir(d):
                if e.is_dir() and not e.name.startswith("."):
                    queue.append((e.path, depth + 1))
        except OSError:
            continue
        scanned += 1
        if scanned % 50 == 0:
            print(f"  ...scanned {scanned} folders")
        if scanned >= 600:
            break

    raise FileNotFoundError(
        "Could not quickly find phase-2-python-sql.zip in your Drive. Put the zip at the "
        "TOP of your Drive (My Drive) and re-run, or set DRIVE_ZIP_PATH at the top of this "
        "cell to its exact path.")


def _find_csv_dir():
    """Return the folder that actually contains olist_orders_dataset.csv."""
    roots = []
    env_dir = os.environ.get("OLIST_DATA_PATH", "")   # set by the pipeline validator
    if env_dir:
        roots.append(env_dir)
    if LOCAL_DATA_DIR:
        roots.append(LOCAL_DATA_DIR)

    if ON_COLAB:
        extract_path = "/content/olist_data"
        # unzip only the first time; reuse the extracted CSVs afterwards
        if not glob.glob(f"{extract_path}/**/olist_orders_dataset.csv", recursive=True):
            zip_path = _colab_find_zip()
            os.makedirs(extract_path, exist_ok=True)
            print(f"Unzipping {os.path.basename(zip_path)} ...")
            with zipfile.ZipFile(zip_path) as z:
                z.extractall(extract_path)
        roots.append(extract_path)
    else:
        # Local: search cwd (recursively) + a few common spots — never the whole
        # home dir (that recursive walk can be very slow). Set LOCAL_DATA_DIR if
        # your CSVs live elsewhere.
        roots += [os.getcwd(),
                  os.path.expanduser("~/Downloads"),
                  os.path.expanduser("~/Desktop"),
                  os.path.expanduser("~/olist")]

    for root in roots:
        if os.path.exists(os.path.join(root, "olist_orders_dataset.csv")):
            return root
        hits = glob.glob(os.path.join(root, "**", "olist_orders_dataset.csv"), recursive=True)
        if hits:
            return os.path.dirname(hits[0])

    raise FileNotFoundError(
        "Olist CSVs not found. Set LOCAL_DATA_DIR (local) or DRIVE_ZIP_PATH (Colab) at "
        "the top of this cell.")


DATA_DIR = _find_csv_dir()
print("Data folder:", DATA_DIR)

# Build a file-based SQLite DB shared by pandas (loading) and jupysql (querying).
DB_PATH = os.environ.get("OLIST_DB_PATH") or (
    "/content/olist.db" if ON_COLAB else os.path.join(tempfile.gettempdir(), "olist.db"))

tables = {
    "orders": "olist_orders_dataset.csv",
    "customers": "olist_customers_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "order_payments": "olist_order_payments_dataset.csv",
    "order_reviews": "olist_order_reviews_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "product_category_translation": "product_category_name_translation.csv",
}

conn = sqlite3.connect(DB_PATH)
for table_name, filename in tables.items():
    df = pd.read_csv(os.path.join(DATA_DIR, filename))
    df.to_sql(table_name, conn, if_exists="replace", index=False)
    print(f"Loaded {table_name}: {len(df):,} rows")
conn.close()
print("\nDatabase ready.")

# On Colab, install jupysql so `%load_ext sql` loads it instead of the legacy
# ipython-sql (see header). Off Colab (local / pipeline validator) jupysql is
# already installed, so we skip the install and stay offline-safe.
if ON_COLAB:
    get_ipython().run_line_magic("pip", "install --quiet --upgrade jupysql")

get_ipython().run_line_magic("load_ext", "sql")

# Guard: if the legacy ipython-sql was already loaded earlier THIS session (e.g.
# an older cell ran first), the freshly installed jupysql cannot hot-swap in — a
# runtime restart is the only fix. jupysql exposes sql.connection.ConnectionManager;
# ipython-sql does not. Stop with a clear instruction instead of a later cryptic
# prettytable KeyError.
import sql.connection as _sqlconn
if not hasattr(_sqlconn, "ConnectionManager"):
    raise RuntimeError(
        "Legacy ipython-sql is active, not jupysql. On Colab: Runtime -> Restart session, "
        "then run THIS setup cell first (before any other cell). Locally: "
        "pip install --upgrade jupysql and restart the kernel."
    )

# Connect the %%sql magic to the SAME database file. autopandas=True is REQUIRED
# (see header). We connect with run_line_magic (not a literal `%sql` line) so the
# computed DB_PATH is interpolated correctly. Do NOT set SqlMagic.style.
get_ipython().run_line_magic("config", "SqlMagic.autopandas = True")
get_ipython().run_line_magic("config", "SqlMagic.feedback = 0")
get_ipython().run_line_magic("sql", f"sqlite:///{DB_PATH}")

# Verify (expected row counts — do not alter without re-running against data):
#   orders 99,441 | customers 99,441 | order_items 112,650 | order_payments 103,886
#   order_reviews 99,224 | products 32,951 | sellers 3,095 | product_category_translation 71


## Question 1 — Orders by status

The `orders` table records the lifecycle status of every order in the `order_status` column. Write **one** `GROUP BY` query that returns each distinct `order_status` alongside how many orders have that status. Alias the count as `n` and sort the results from the most common status to the least common.

**Expected:** 8 rows — the top row is `delivered` with 96,478 orders (then shipped 1,107 | canceled 625 | unavailable 609 | invoiced 314 | processing 301 | created 5 | approved 2)

In [ ]:
%%sql q1 <<
-- Your query here

In [ ]:
# --- CHECK Q1 — do not edit ---
assert q1.shape[0] == 8, f"Q1: expected 8 status groups, got {q1.shape[0]}"
assert q1.iloc[0]['order_status'] == 'delivered', "Q1: top row should be 'delivered' — sort by the count descending"
assert int(q1.iloc[0]['n']) == 96478, "Q1: expected 96,478 delivered orders"
print("✅ Q1 correct")
q1  # show the result of your query

## Question 2 — Isolate the shipped group with HAVING

Start from the same grouped query as Question 1, but this time keep **only** the `shipped` group by filtering the groups with a `HAVING` clause on `order_status`. Return the count aliased as `n`.

**Expected:** 1,107 shipped orders

In [ ]:
%%sql q2 <<
-- Your query here

In [ ]:
# --- CHECK Q2 — do not edit ---
assert int(q2.iloc[0]['n']) == 1107, "Q2: expected 1,107 shipped orders"
print("✅ Q2 correct")
q2  # show the result of your query

## Question 3 — Payment summary by payment type

The `order_payments` table has one row per payment transaction, with a `payment_type` and a `payment_value`. Group it by `payment_type` and return, for each type:

- the number of transactions, aliased as `n`
- the average payment value rounded to 2 decimal places, aliased as `avg_value`
- the total payment value rounded to 2 decimal places, aliased as `total_value`

Sort so the payment type with the most transactions comes first.

**Expected:** the top row is `credit_card` with 76,795 transactions, an average of 163.32 and a total of 12,542,084.19

In [ ]:
%%sql q3 <<
-- Your query here

In [ ]:
# --- CHECK Q3 — do not edit ---
assert q3.iloc[0]['payment_type'] == 'credit_card', "Q3: top row should be 'credit_card' — sort by the transaction count descending"
assert int(q3.iloc[0]['n']) == 76795, "Q3: expected 76,795 credit_card transactions"
assert round(float(q3.iloc[0]['avg_value']), 2) == 163.32, "Q3: expected a credit_card average payment value of 163.32"
assert round(float(q3.iloc[0]['total_value']), 2) == 12542084.19, "Q3: expected a credit_card total payment value of 12,542,084.19"
print("✅ Q3 correct")
q3  # show the result of your query

## Question 4 — The single biggest customer state

The `customers` table has a `customer_state` column (two-letter Brazilian state codes). Which **one** state has the most customers? Group by `customer_state`, count the customers as `n`, sort from the largest group down, and return only the top row.

**Expected:** SP with 41,746 customers

In [ ]:
%%sql q4 <<
-- Your query here

In [ ]:
# --- CHECK Q4 — do not edit ---
assert q4.shape[0] == 1, f"Q4: expected exactly 1 row (use LIMIT 1), got {q4.shape[0]}"
assert q4.iloc[0]['customer_state'] == 'SP', "Q4: expected SP to be the largest customer state"
assert int(q4.iloc[0]['n']) == 41746, "Q4: expected 41,746 customers in SP"
print("✅ Q4 correct")
q4  # show the result of your query

## Question 5 — Only the high-volume payment types

Using `order_payments` again, group by `payment_type` and count the transactions as `n`, but keep only the payment types with **more than 5,000** transactions. This is a job for `HAVING`, not `WHERE` — the count only exists after the rows have been grouped. Sort from the most transactions down.

**Expected:** 3 rows — credit_card 76,795 | boleto 19,784 | voucher 5,775

In [ ]:
%%sql q5 <<
-- Your query here

In [ ]:
# --- CHECK Q5 — do not edit ---
assert q5.shape[0] == 3, f"Q5: expected 3 payment types above 5,000 transactions, got {q5.shape[0]}"
assert q5.iloc[0]['payment_type'] == 'credit_card', "Q5: top row should be 'credit_card' — sort by the count descending"
assert int(q5.iloc[0]['n']) == 76795, "Q5: expected 76,795 credit_card transactions"
assert q5.iloc[2]['payment_type'] == 'voucher', "Q5: the third row should be 'voucher'"
assert int(q5.iloc[2]['n']) == 5775, "Q5: expected 5,775 voucher transactions"
print("✅ Q5 correct")
q5  # show the result of your query